# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished: {getattr(metadata, 'datePublished', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant metadata describes sets of records (tables) called record sets. Each record set has an `@id` and a set of fields, each with its own `@id`.

In [ ]:
# Explore record sets with field and column information
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets are defined in the dataset metadata. The dataset might define record sets within file objects or in the distribution section.")

data_distributions = getattr(metadata, 'distribution', [])
if not record_sets and data_distributions:
    # Try to extract recordSet from each distribution (this is common Croissant pattern)
    for dist in data_distributions:
        # Access each DataFileObject/recordSet
        file_obj = dist
        # Using mlcroissant API's details
        print("Distribution:", file_obj["@id"])

In [ ]:
# List all record set IDs discoverable by the mlcroissant API
for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"    Field @id: {f['@id']}, name: {f.get('name', '')}, dataType: {f.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> :information_source: **All entity selection below is by their `@id` as per the Croissant schema specification.**

Let's extract each record set available using its `@id`. We'll display the columns for the first record set for reference.

In [ ]:
# Get the list of all recordSet @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
# Dictionary of DataFrames, indexed by recordSet @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")

# Display the columns for the first recordset
if record_set_ids:
    print(f"\nColumns for record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Below, we select a numeric field from the first record set for demonstration. Adjust variables as needed based on the actual field `@id`s discovered above.

In [ ]:
# Pick first record set
if len(record_set_ids) > 0:
    example_record_set_id = record_set_ids[0]
    df = dataframes[example_record_set_id]
    # Try to pick a numeric field
    possible_numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if not possible_numeric_fields:
        # Try to convert columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        possible_numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        fnorm = f"{numeric_field}_normalized"
        filtered_df[fnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, fnorm]].head())
        # Try to group by another field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field:
                # Choose a likely categorical field
                if df[col].nunique() < df.shape[0] // 2:
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped filtered records by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the record set. Please review the field IDs above and adjust for your analysis.")
else:
    print("No record sets available to demonstrate EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib. Adjust field IDs as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_set_ids) > 0 and possible_numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouping is possible
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print("Nothing to plot: no numeric field detected.")

## 6. Conclusion
In this notebook, we demonstrated how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, overview, extract, and analyze tabular data described by a Croissant schema using strict entity referencing by `@id`. Further analysis can be performed by refining selection of fields or exploring more record sets as specified in the metadata.

> **Note:** Refer to the original Croissant schema for full details on each field, record set, and variable definition. Update the notebook with specific `@id`s for field-level analysis as required for your research.

Happy experimenting!